In [ ]:
import time
from datetime import datetime
import pandas as pd
import re
from apscheduler.schedulers.background import BackgroundScheduler
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
driver = webdriver.Chrome(options=chrome_options)

URL = "https://www.sothailand.org/genws/dashboard"
data_history = []

def save_data():
    if data_history:
        df = pd.DataFrame(data_history)
        df.to_csv("scraped_power_data.csv", index=False)
        return df
    return None

def scrape_and_store():
    driver.get(URL)
    time.sleep(3)
    wait = WebDriverWait(driver, 5)

    date_element = driver.find_elements(By.XPATH, "//div[contains(@class, 'date')]//span[1]")
    date_str = date_element[0].text if date_element else datetime.now().strftime("%d-%m-%Y")

    time_element = driver.find_elements(By.XPATH, "//div[contains(@class, 'time')]//span")
    time_str = time_element[0].text if time_element else datetime.now().strftime("%H:%M")

    power_element = driver.find_elements(By.XPATH, "//span[contains(@style,'font-size')]")
    if power_element:
        current_value = power_element[0].text.split()[0].replace(',', '')
    else:
        elements = driver.find_elements(By.XPATH, "//div[contains(text(), 'MW')]")
        if elements:
            numbers = re.findall(r'[\d,]+\.?\d*', elements[0].text)
            current_value = numbers[0].replace(',', '') if numbers else "0"
        else:
            current_value = "0"

    temp_element = driver.find_elements(By.XPATH, "//span[contains(text(), '°C')]")
    temp = temp_element[0].text.replace('°C', '').strip() if temp_element else "0"

    now = datetime.now()
    power_value = float(current_value) if current_value.replace('.', '', 1).isdigit() else 0.0
    temp_value = float(temp) if temp.replace('.', '', 1).isdigit() else 0.0

    data = {
        'scrape_time': now,
        'display_date': date_str,
        'display_time': time_str,
        'current_value_MW': power_value,
        'temperature_C': temp_value,
    }

    data_history.append(data)

    if len(data_history) % 5 == 0:
        save_data()

if __name__ == "__main__":
    sched = BackgroundScheduler()
    sched.add_job(scrape_and_store, 'interval', minutes=1)
    sched.start()

    try:
        scrape_and_store()
        while True:
            time.sleep(60)
            if len(data_history) % 10 == 0:
                save_data()
    except KeyboardInterrupt:
        print("Shutting down...")
    finally:
        save_data()
        sched.shutdown()
        driver.quit()

In [ ]:
import time
from datetime import datetime
import pandas as pd
import re
from apscheduler.schedulers.background import BackgroundScheduler
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import logging

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filename='scraper_log.log'
)

# Configure Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--enable-logging")
chrome_options.add_argument("--v=1")  # Verbose logging for Chrome

# Add capability to capture console logs
chrome_options.set_capability('goog:loggingPrefs', {'browser': 'ALL'})

# Initialize driver with better error handling
try:
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )
    logging.info("Chrome driver initialized successfully")
except Exception as e:
    logging.error(f"Failed to initialize Chrome driver: {e}")
    raise

URL = "https://www.sothailand.org/genws/dashboard"
data_history = []
csv_filename = "scraped_power_data.csv"

def save_data():
    """Save the collected data to a CSV file."""
    if data_history:
        df = pd.DataFrame(data_history)
        df.to_csv(csv_filename, index=False)
        logging.info(f"Data saved to {csv_filename} - {len(data_history)} records")
        print(f"Data saved - {len(data_history)} records")
        return df
    logging.warning("No data to save")
    return None

def scrape_and_store():
    """Scrape the website and extract power data from console logs."""
    try:
        # Navigate to the URL
        driver.get(URL)
        logging.info(f"Navigated to {URL}")
        
        # Wait for the page to fully load and JavaScript to execute
        time.sleep(10)
        
        # Get the browser console logs
        logs = driver.get_log('browser')
        logging.info(f"Retrieved {len(logs)} console log entries")
        
        # Extract the most recent updateMessageArea entry
        latest_data = None
        for log in reversed(logs):  # Process from newest to oldest
            message = log.get('message', '')
            if 'updateMessageArea' in message:
                # Extract the format: updateMessageArea: 13200 , 03:40 , 21,038.1 , 26.8
                pattern = r'updateMessageArea:\s*(\d+)\s*,\s*(\d{2}:\d{2})\s*,\s*([\d,]+\.\d+)\s*,\s*([\d\.]+)'
                matches = re.search(pattern, message)
                
                if matches:
                    timestamp = matches.group(1)
                    display_time = matches.group(2)
                    power_value = float(matches.group(3).replace(',', ''))
                    temp_value = float(matches.group(4))
                    
                    # Get current date
                    now = datetime.now()
                    display_date = now.strftime("%d-%m-%Y")
                    
                    latest_data = {
                        'scrape_time': now,
                        'display_date': display_date,
                        'display_time': display_time,
                        'current_value_MW': power_value,
                        'temperature_C': temp_value,
                    }
                    
                    data_history.append(latest_data)
                    logging.info(f"Scraped data: Time={display_time}, MW={power_value}, Temp={temp_value}°C")
                    print(f"Scraped: {display_time}, MW: {power_value}, Temp: {temp_value}°C")
                    break
        
        # If no console log data was found, try to scrape from DOM as fallback
        if not latest_data:
            logging.warning("No updateMessageArea data found in console logs, trying DOM scraping")
            
            # Attempt to extract from the JavaScript code shown in your screenshot
            # Looking for elements where the MW value might be displayed
            try:
                page_source = driver.page_source
                # Look for values with MW label
                mw_pattern = r'([\d,]+\.\d+)\s*MW'
                temp_pattern = r'([\d\.]+)\s*°C'
                time_pattern = r'(\d{2}:\d{2})'
                
                mw_match = re.search(mw_pattern, page_source)
                temp_match = re.search(temp_pattern, page_source)
                time_match = re.search(time_pattern, page_source)
                
                now = datetime.now()
                display_date = now.strftime("%d-%m-%Y")
                display_time = time_match.group(1) if time_match else now.strftime("%H:%M")
                
                power_value = float(mw_match.group(1).replace(',', '')) if mw_match else 0.0
                temp_value = float(temp_match.group(1)) if temp_match else 0.0
                
                latest_data = {
                    'scrape_time': now,
                    'display_date': display_date,
                    'display_time': display_time,
                    'current_value_MW': power_value,
                    'temperature_C': temp_value,
                }
                
                data_history.append(latest_data)
                logging.info(f"Fallback scrape: Time={display_time}, MW={power_value}, Temp={temp_value}°C")
                print(f"Fallback: {display_time}, MW: {power_value}, Temp: {temp_value}°C")
            except Exception as e:
                logging.error(f"Fallback scraping failed: {e}")
                
    except Exception as e:
        logging.error(f"Error during scraping: {e}")
        print(f"Error during scraping: {e}")

if __name__ == "__main__":
    # Initialize the scheduler
    sched = BackgroundScheduler()
    sched.add_job(scrape_and_store, 'interval', minutes=5)  # Scrape every 5 minutes
    sched.start()
    logging.info("Scheduler started with 5-minute intervals")

    try:
        print("Starting power consumption monitoring...")
        logging.info("Starting power consumption monitoring")
        scrape_and_store()  # Initial scrape
        
        # Main loop to keep the program running and save periodically
        while True:
            time.sleep(60)  # Check every minute
            current_records = len(data_history)
            if current_records > 0 and current_records % 12 == 0:  # Save every ~hour (12 x 5min scrapes)
                save_data()
                
    except KeyboardInterrupt:
        print("Shutting down...")
        logging.info("Shutdown initiated by user")
    except Exception as e:
        print(f"Unexpected error: {e}")
        logging.error(f"Unexpected error: {e}")
    finally:
        save_data()
        sched.shutdown()
        driver.quit()
        logging.info("Scraper shutdown complete")

Starting power consumption monitoring...
Fallback: 03:46, MW: 0.0, Temp: 0.0°C
Fallback: 03:51, MW: 0.0, Temp: 0.0°C
Shutting down...
Data saved - 2 records


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import datetime
import re
import csv
import os

# File to save the data
CSV_FILENAME = 'egat_power_data.csv'

def scrape_egat_data():
    """Scrape real-time data from EGAT website"""
    url = 'https://www.sothailand.com/sysgen/egat/'
    
    try:
        # Send request to the website
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for HTTP errors
        
        # Parse HTML content
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Extract the current time of scraping
        scrape_time = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        # Find the message area content - adjust the selector as needed based on actual website structure
        # This is a placeholder - you'll need to inspect the page and identify the correct elements
        message_area = soup.find('div', {'id': 'messageArea'}) or soup.find('div', {'class': 'messageArea'})
        
        if message_area:
            message_text = message_area.text.strip()
            
            # Use regex to extract the required information
            # Pattern may need adjustment based on actual format
            pattern = r'(\d+)\s*,\s*(\d+:\d+)\s*,\s*([\d,\.]+)\s*,\s*([\d\.]+)'
            match = re.search(pattern, message_text)
            
            if match:
                message_id = match.group(1)
                display_time = match.group(2)
                current_value_mw = match.group(3).replace(',', '')  # Remove commas from number
                temperature_c = match.group(4)
                
                # Get current date for display_date
                display_date = datetime.datetime.now().strftime('%Y-%m-%d')
                
                # Return as a dictionary
                return {
                    'scrape_time': scrape_time,
                    'display_date': display_date,
                    'display_time': display_time,
                    'current_value_MW': current_value_mw,
                    'temperature_C': temperature_c
                }
        
        print(f"Failed to extract data. Website structure may have changed.")
        return None
        
    except Exception as e:
        print(f"Error scraping data: {e}")
        return None

def save_to_csv(data):
    """Save the scraped data to CSV file"""
    file_exists = os.path.isfile(CSV_FILENAME)
    
    with open(CSV_FILENAME, 'a', newline='') as csvfile:
        fieldnames = ['scrape_time', 'display_date', 'display_time', 'current_value_MW', 'temperature_C']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        if not file_exists:
            writer.writeheader()
            
        writer.writerow(data)

def load_to_pandas():
    """Load the CSV data into a pandas DataFrame"""
    try:
        df = pd.read_csv(CSV_FILENAME)
        return df
    except Exception as e:
        print(f"Error loading data to pandas: {e}")
        return pd.DataFrame()

def main():
    """Main function to run the scraper every 5 minutes"""
    print("Starting EGAT power data scraper...")
    print(f"Data will be saved to {CSV_FILENAME}")
    
    try:
        while True:
            # Scrape data
            data = scrape_egat_data()
            
            if data:
                # Save to CSV
                save_to_csv(data)
                
                # Load and display current pandas DataFrame
                df = load_to_pandas()
                print("\nCurrent DataFrame:")
                print(df.tail())
                
                print(f"\nSuccessfully scraped data at {data['scrape_time']}")
                print(f"Current power: {data['current_value_MW']} MW, Temperature: {data['temperature_C']}°C")
            else:
                print("Failed to scrape data.")
                
            # Wait for 5 minutes before the next scrape
            print("\nWaiting 5 minutes for next scrape...")
            time.sleep(300)  # 300 seconds = 5 minutes
            
    except KeyboardInterrupt:
        print("\nScraping stopped by user.")
        
        # Load and display final pandas DataFrame
        df = load_to_pandas()
        print("\nFinal DataFrame:")
        print(df)

if __name__ == "__main__":
    main()

Starting EGAT power data scraper...
Data will be saved to egat_power_data.csv
Failed to extract data. Website structure may have changed.
Failed to scrape data.

Waiting 5 minutes for next scrape...
Failed to extract data. Website structure may have changed.
Failed to scrape data.

Waiting 5 minutes for next scrape...


In [ ]:
import time
import datetime
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import os
import re
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class EGATRealTimeScraper:
    def __init__(self, url="https://www.sothailand.com/sysgen/egat/", output_file="egat_power_data.csv"):
        self.url = url
        self.output_file = output_file
        self.driver = None
        self._initialize_driver()
        
    def _initialize_driver(self):
        chrome_options = Options()
        chrome_options.add_argument("--headless")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--log-level=0")
        chrome_options.set_capability('goog:loggingPrefs', {'browser': 'ALL'})
        
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=chrome_options)

    def extract_data_from_console(self):
        if not self.driver:
            return None
            
        logs = self.driver.get_log('browser')
        found_data = None

        for log_entry in logs:
            message = log_entry.get('message', '')
            if 'updateMessageArea:' in message:
                match = re.search(r'updateMessageArea:\s+(\d+)\s*,\s*(\d+:\d+)\s*,\s*([\d,]+\.\d+)\s*,\s*(\d+\.\d+)', message)
                if match:
                    display_date_id = match.group(1).strip()
                    display_time = match.group(2).strip()
                    current_value_mw = float(match.group(3).replace(',', '').strip())
                    temperature_c = float(match.group(4).strip())

                    found_data = {
                        'scrape_time': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                        'display_date_id': display_date_id,
                        'display_time': display_time,
                        'current_value_MW': current_value_mw,
                        'temperature_C': temperature_c
                    }
        return found_data

    def save_to_csv(self, new_data):
        if not new_data:
            return None

        df_new = pd.DataFrame([new_data])
        
        if os.path.exists(self.output_file):
            df_new.to_csv(self.output_file, mode='a', header=False, index=False, encoding='utf-8')
        else:
            df_new.to_csv(self.output_file, index=False, encoding='utf-8')
            
        return df_new

    def scrape_once(self):
        if self.driver is None:
            self._initialize_driver()
            
        self.driver.get(self.url)
        time.sleep(5)  # Reduced wait time for faster performance
        
        data = self.extract_data_from_console()
        
        if data:
            self.save_to_csv(data)
        
        return data

    def scrape_continuously(self, interval_minutes=5):
        if interval_minutes <= 0:
            return
            
        while True:
            start_time = time.time()
            self.scrape_once()
            
            elapsed_time = time.time() - start_time
            sleep_time = max(0, (interval_minutes * 60) - elapsed_time)
            
            if sleep_time > 0:
                time.sleep(sleep_time)

    def close(self):
        if self.driver:
            self.driver.quit()
            self.driver = None

if __name__ == "__main__":
    scraper = EGATRealTimeScraper(output_file="egat_realtime_power.csv")
    scraper.scrape_continuously(interval_minutes=5)
    scraper.close()

2025-05-13 11:05:52,281 - INFO - ====== WebDriver manager ======
2025-05-13 11:05:53,403 - INFO - Get LATEST chromedriver version for google-chrome
2025-05-13 11:05:53,451 - INFO - Get LATEST chromedriver version for google-chrome
2025-05-13 11:05:53,500 - INFO - Driver [C:\Users\kongl\.wdm\drivers\chromedriver\win64\136.0.7103.92\chromedriver-win32/chromedriver.exe] found in cache
